

# Importing modules and settings

### Importing Libraries

In [ ]:
import numpy as np
import pandas  as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import seaborn as sns
import os

In [ ]:
import gzip
import fnmatch
import re
from scipy.sparse import csr_matrix
import anndata as ad

In [ ]:
from matplotlib.colors import ListedColormap

### General settings of Scanpy

In [ ]:
sc.settings.verbosity = 4
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

In [ ]:
# Create a CMAP for the UMAP plotting
umap_cmap = sns.blend_palette(['xkcd:light grey', 'xkcd:indigo'], as_cmap = True)

In [ ]:
# Declare the output files
name_of_analysis = 'Smed_L47_Subclustering'
name_of_subclustering = '_4CnCL_'
results_file = name_of_analysis + name_of_subclustering + 'Results.h5ad'



# input file

In [ ]:
adata = sc.read_h5ad('./Smed_L78-L47_20250523_Annotated.h5ad')

In [ ]:
adata

In [ ]:
adatar = adata.raw.to_adata()

In [ ]:
adatar

# filter cells

In [ ]:
adataf = adatar[(adatar.obs['Condition_2C/4C'] == '4C')
               # & (adatar.obs['neoblast_score'] > 0.185)
                & (adatar.obs['broad_names'] == 'neoblasts')
               ].copy()

In [ ]:
adataf

# Scaling the data

In [ ]:
# Perform data scaling with z-score standardisation
sc.pp.scale(adataf)

# Performing the PCA and kNN analysis

In [ ]:
# Perform PCA analysis
sc.tl.pca(adataf, svd_solver='arpack', n_comps = 150)

In [ ]:
# Plot the proportion of variance explained by each PC
sc.pl.pca_variance_ratio(adataf, n_pcs=150, log=True)

In [ ]:
# Build the kNN tree
sc.pp.neighbors(adataf, n_neighbors=30, n_pcs=110)

In [ ]:
# Compute UMAP and plot
sc.tl.umap(adataf, min_dist= 0.75, spread = 1.25, alpha = 1, gamma = 2)

In [ ]:
sc.pl.umap(adataf)

In [ ]:
# UMAP with expression of Smedwi1
with plt.rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adataf, color='h1SMcG0013999', size = 30, color_map = umap_cmap)

# Clustering

In [ ]:
# Select resolutions 
resolutions = [1, 2, 3, 4, 5]

In [ ]:
# Run Leiden for each of the above resolution
for i in resolutions:
    sc.tl.leiden(adataf, resolution = i, key_added = 'leiden'+ name_of_subclustering + str(i))
    sc.pl.umap(adataf, color= 'leiden'+ name_of_subclustering + str(i))

In [ ]:
results = pd.DataFrame(
    0,
    index = adata.obs['annotated_names'].cat.categories,
    columns = resolutions,
    dtype = int
)

In [ ]:
results

In [ ]:
for i in resolutions:
    for l in adataf.obs['leiden'+ name_of_subclustering + str(i)].cat.categories:
        counts = adataf.obs.loc[(adataf.obs['leiden'+ name_of_subclustering + str(i)] == l), 'annotated_names'].value_counts().sort_values(ascending = False)
        if counts[0] > (counts[1:].sum() / 10):
            results.loc[counts.index[0], i] += 1

In [ ]:
results

In [ ]:
results.max().max()

In [ ]:
sns.color_palette(palette='RdYlBu', n_colors = 10)

In [ ]:
scale = sns.color_palette(palette='RdYlBu', n_colors = results.max().max())
cmap_colors = [(0, 0, 0)] + scale
cmap = ListedColormap(cmap_colors)


In [ ]:
fig, ax = plt.subplots(figsize=(15,30))
sns.heatmap(results,cmap = cmap, ax=ax)

# Output files

In [ ]:
 adataf.write(results_file)